# 3장 예제 — 3.17 실전 정제 ② 통계 규칙이 못 잡는 물리적 불가값 (필수, 25분)

- **교재 출처**: manuscript/31_ch3_eda_fe.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 14_3-14_EDA_리포트_정리.py, 15_3-15_연습문제.py, 16_3-16_실전_정제_①_현장_엑셀_워크북_해체하기.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd

### 이 절에서 쓸 데이터 — 18650 셀 수명 로그

In [ ]:
rul = pd.read_csv('data/rul/battery_rul_original.csv')
print("모양:", rul.shape)
print("결측:", int(rul.isna().sum().sum()), "건 · 중복 행:", int(rul.duplicated().sum()), "건")
print(list(rul.columns))

# 교재 실행 결과 ------------------------------------------------------
#   모양: (15064, 9)
#   결측: 0 건 · 중복 행: 0 건
#   ['Cycle_Index', 'Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Max. Voltage Dischar. (V)', 'Min. Voltage Charg. (V)', 'Time at 4.15V (s)', 'Time constant current (s)', 'Charging time (s)', 'RUL']

### describe() 한 줄이 던지는 경고

In [ ]:
t = ['Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Time at 4.15V (s)']
print(rul[t].describe().loc[['min', '50%', 'max']].round(1).to_string())

# 교재 실행 결과 ------------------------------------------------------
#        Discharge Time (s)  Decrement 3.6-3.4V (s)  Time at 4.15V (s)
#   min                 8.7               -397645.9             -113.6
#   50%              1557.2                   439.2             2930.2
#   max            958320.4                406703.8           245101.1

### IQR도 z-score도 이것을 잡지 못한다

In [ ]:
s = rul['Time at 4.15V (s)']
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR 경계: {lo:.1f} ~ {hi:.1f}")
print("IQR 규칙 검출:", int(((s < lo) | (s > hi)).sum()), "건")

z = (s - s.mean()) / s.std()
print("|z| > 3 검출:", int((z.abs() > 3).sum()), "건")
print("실제 음수:", int((s < 0).sum()), "건 (최솟값", round(float(s.min()), 1), "초)")
print("음수 중 IQR 규칙이 잡은 것:", int(((s < 0) & ((s < lo) | (s > hi))).sum()), "건")
print("음수 중 |z|>3 이 잡은 것 :", int(((s < 0) & (z.abs() > 3)).sum()), "건")

# 교재 실행 결과 ------------------------------------------------------
#   IQR 경계: -1560.3 ~ 7477.5
#   IQR 규칙 검출: 167 건
#   |z| > 3 검출: 167 건
#   실제 음수: 9 건 (최솟값 -113.6 초)
#   음수 중 IQR 규칙이 잡은 것: 0 건
#   음수 중 |z|>3 이 잡은 것 : 0 건

### 도메인 규칙 다섯 줄이 이긴다

In [ ]:
rules = {
    '음수 시간 · 전압강하 구간': rul['Decrement 3.6-3.4V (s)'] < 0,
    '음수 시간 · 4.15V 도달': rul['Time at 4.15V (s)'] < 0,
    '부분 > 전체 · 4.15V 도달 > 총 충전': rul['Time at 4.15V (s)'] > rul['Charging time (s)'],
    '부분 > 전체 · 전압강하 > 총 방전':
        rul['Decrement 3.6-3.4V (s)'].abs() > rul['Discharge Time (s)'],
    '부분 > 전체 · CC구간 > 총 충전':
        rul['Time constant current (s)'] > rul['Charging time (s)'],
}

bad = pd.Series(False, index=rul.index)
for name, m in rules.items():
    print(f"{name}: {int(m.sum())}건")
    bad = bad | m

print("---")
print("합집합:", int(bad.sum()), "행 =", round(100 * float(bad.mean()), 2), "%")
print("제거 후:", rul[~bad].shape)

# 교재 실행 결과 ------------------------------------------------------
#   음수 시간 · 전압강하 구간: 24건
#   음수 시간 · 4.15V 도달: 9건
#   부분 > 전체 · 4.15V 도달 > 총 충전: 28건
#   부분 > 전체 · 전압강하 > 총 방전: 34건
#   부분 > 전체 · CC구간 > 총 충전: 0건
#   ---
#   합집합: 68 행 = 0.45 %
#   제거 후: (14996, 9)

### ID 컬럼이 없는 로그에서 로트를 복원하기

In [ ]:
reset = rul['Cycle_Index'].diff() < 0
print("사이클 번호가 되돌아간 지점:", int(reset.sum()), "곳")

rul['cell_id'] = reset.cumsum() + 1
print("복원된 셀 개수:", rul['cell_id'].nunique())

# 교재 실행 결과 ------------------------------------------------------
#   사이클 번호가 되돌아간 지점: 13 곳
#   복원된 셀 개수: 14

In [ ]:
summ = rul.groupby('cell_id').agg(행수=('Cycle_Index', 'size'),
                                  EOL=('Cycle_Index', 'max'))
summ['EOL'] = summ['EOL'].astype(int)
summ['불가값행'] = bad.groupby(rul['cell_id']).sum()
print(summ.head(5).to_string())
print("행수:", int(summ['행수'].min()), "~", int(summ['행수'].max()),
      "· EOL:", int(summ['EOL'].min()), "~", int(summ['EOL'].max()))
print("불가값이 하나도 없는 셀:", int((summ['불가값행'] == 0).sum()), "개")

# 교재 실행 결과 ------------------------------------------------------
#              행수   EOL  불가값행
#   cell_id                  
#   1        1076  1113     2
#   2        1079  1108     2
#   3        1077  1108     3
#   4        1081  1108     3
#   5        1077  1134     5
#   행수: 1051 ~ 1081 · EOL: 1103 ~ 1134
#   불가값이 하나도 없는 셀: 0 개

In [ ]:
gap = summ['EOL'] - summ['행수']
print("셀별 누락 사이클:", int(gap.min()), "~", int(gap.max()), "· 합계:", int(gap.sum()))

# 교재 실행 결과 ------------------------------------------------------
#   셀별 누락 사이클: 25 ~ 61 · 합계: 481

### 복원한 셀 ID가 곧바로 드러낸 것 — 타깃 누수 예고

In [ ]:
rul['EOL'] = rul.groupby('cell_id')['Cycle_Index'].transform('max')
print("RUL == EOL − Cycle_Index 가 15,064행 전부에서 성립? ",
      bool((rul['RUL'] == rul['EOL'] - rul['Cycle_Index']).all()))
print("corr(Cycle_Index, RUL) =", round(float(rul['Cycle_Index'].corr(rul['RUL'])), 6))

# 교재 실행 결과 ------------------------------------------------------
#   RUL == EOL − Cycle_Index 가 15,064행 전부에서 성립?  True
#   corr(Cycle_Index, RUL) = -0.999756